# Elo ratings, 2023-24 to 2025-26 (corrected)

This replaces `elo_from_23.ipynb`. It continues FiveThirtyEight's NBA Elo from the end of the 2022-23 Finals, and it writes `single_game_with_elo_corrected.csv`.

**What changed from the first version**

| | First version | This version |
|---|---|---|
| Margin-of-victory formula | NFL version, `ln(MOV + 1) * 2.2 / (2.2 + 0.001 * gap)` | NBA version, `(MOV + 3)^0.8 / (7.5 + 0.006 * gap)` |
| Starting ratings | Typed in, did not match FiveThirtyEight, averaged 1533 | FiveThirtyEight's actual post-Finals ratings, average 1505 |
| Games counted | Regular season only | Regular season, play-in and playoffs |
| Neutral-site games | Skipped (10 games) | Counted, with no home-court bonus |

**Rules kept from FiveThirtyEight:** K = 20, 100-point home-court bonus, and between seasons 75% of the old rating plus 25% of 1505.

**Source of the starting ratings:** FiveThirtyEight's `nba_elo.csv`, archived at web.archive.org on 2023-10-24 (the original link no longer works). These are each team's `elo_post` after its last game of 2022-23.

In [1]:
import pandas as pd
import numpy as np

K = 20
HOME_ADVANTAGE = 100
LEAGUE_MEAN = 1505
CARRYOVER = 0.75

## 1. Starting ratings (after the 2023 Finals)

In [2]:
elo_2223_final = {
    "DEN": 1685.23, "BOS": 1663.59, "PHI": 1640.10, "MIA": 1605.08, "NYK": 1595.16,
    "LAL": 1591.53, "GSW": 1590.70, "CLE": 1569.96, "MEM": 1568.28, "MIL": 1561.36,
    "TOR": 1560.16, "PHX": 1557.15, "CHI": 1543.22, "SAC": 1537.09, "ATL": 1531.15,
    "NOP": 1529.82, "LAC": 1523.62, "MIN": 1519.82, "OKC": 1516.83, "BKN": 1491.32,
    "ORL": 1448.47, "DAL": 1442.30, "WAS": 1436.36, "UTA": 1414.42, "IND": 1399.75,
    "CHA": 1386.72, "SAS": 1329.73, "HOU": 1328.38, "POR": 1310.23, "DET": 1271.09,
}

print("Teams:", len(elo_2223_final))
print("League average:", round(np.mean(list(elo_2223_final.values())), 2))

Teams: 30
League average: 1504.95


## 2. Elo functions

In [3]:
def win_prob(home_elo, away_elo, neutral=False):
    """Chance the home (or first-listed) team wins."""
    bonus = 0 if neutral else HOME_ADVANTAGE
    return 1 / (1 + 10 ** (-(home_elo + bonus - away_elo) / 400))


def mov_multiplier(margin, winner_gap):
    """NBA margin-of-victory multiplier. winner_gap is winner's Elo minus loser's, including home court."""
    return (abs(margin) + 3) ** 0.8 / (7.5 + 0.006 * winner_gap)


def elo_change(home_elo, away_elo, margin, neutral=False):
    """Points gained by the home team. The away team loses the same amount."""
    bonus = 0 if neutral else HOME_ADVANTAGE
    home_won = margin > 0
    gap = home_elo + bonus - away_elo
    winner_gap = gap if home_won else -gap
    result = 1 if home_won else 0
    return K * mov_multiplier(margin, winner_gap) * (result - win_prob(home_elo, away_elo, neutral))

### Test the formula against known answers

These are the five 2023 Finals games as published by FiveThirtyEight. If the formula is right, it must reproduce their post-game ratings. **The first version's formula fails this test.**

In [4]:
# (team1 pre-game Elo, team2 pre-game Elo, team1 margin, team1 post-game Elo)
known_games = [
    (1649.957428, 1640.358994,  11, 1656.989505),
    (1656.989505, 1633.326917,  -3, 1648.665508),
    (1641.650915, 1648.665508, -15, 1623.302940),
    (1623.302940, 1667.013482, -13, 1608.411644),
    (1681.904778, 1608.411644,   5, 1685.231861),
]

for pre1, pre2, margin, published in known_games:
    calculated = pre1 + elo_change(pre1, pre2, margin)
    print(f"calculated {calculated:.3f} | published {published:.3f}")
    assert abs(calculated - published) < 0.01

print("PASS: formula reproduces FiveThirtyEight's published ratings")

calculated 1656.990 | published 1656.990
calculated 1648.666 | published 1648.666
calculated 1623.303 | published 1623.303
calculated 1608.412 | published 1608.412
calculated 1685.232 | published 1685.232
PASS: formula reproduces FiveThirtyEight's published ratings


## 3. Put every game in one table

Three sources, one row per game:

- `single_game.csv`: regular-season games, from the home team's side
- `rest_travel.csv`: the 10 neutral-site games that `single_game.csv` leaves out
- `postseason_game_logs_2024_2025.csv`: 2024 and 2025 play-in and playoff games, downloaded with `nba_api` (`LeagueGameLog`, `season_type_all_star` = `"PlayIn"` and `"Playoffs"`)

The 2026 playoffs aren't needed. They come after the last game in our file, so they could only affect 2026-27 ratings.

In [5]:
single_game = pd.read_csv("single_game.csv")
rest_travel = pd.read_csv("rest_travel.csv")
postseason = pd.read_csv("postseason_game_logs_2024_2025.csv")

columns = ["GAME_ID", "SEASON", "GAME_DATE", "team1", "team2", "margin", "neutral", "stage"]

# Regular-season games
regular = single_game.rename(columns={
    "TEAM_ABBREVIATION_home": "team1",
    "TEAM_ABBREVIATION_away": "team2",
    "PLUS_MINUS": "margin",
}).assign(neutral=False, stage="Regular")[columns]

# Neutral-site games: in rest_travel.csv, but with no row in single_game.csv
neutral_rows = rest_travel[~rest_travel["GAME_ID"].isin(single_game["GAME_ID"])].sort_values(["GAME_ID", "TEAM_ABBREVIATION"])
neutral_first = neutral_rows.groupby("GAME_ID").first().reset_index()
neutral_last = neutral_rows.groupby("GAME_ID").last().reset_index()
neutral = neutral_first.assign(
    team1=neutral_first["TEAM_ABBREVIATION"],
    team2=neutral_last["TEAM_ABBREVIATION"],
    margin=neutral_first["PLUS_MINUS"],
    neutral=True,
    stage="Neutral",
)[columns]

# Play-in and playoff games: the home team is the row with "vs." in MATCHUP
post_home = postseason[postseason["MATCHUP"].str.contains("vs.", regex=False)]
post_away = postseason[postseason["MATCHUP"].str.contains("@", regex=False)]
post = post_home.merge(post_away, on="GAME_ID", suffixes=("_home", "_away"), validate="one_to_one")
post = post.assign(
    SEASON=post["SEASON_home"],
    GAME_DATE=post["GAME_DATE_home"],
    team1=post["TEAM_ABBREVIATION_home"],
    team2=post["TEAM_ABBREVIATION_away"],
    margin=post["PLUS_MINUS_home"],
    neutral=False,
    stage=post["SEASON_TYPE_home"],
)[columns]

all_games = pd.concat([regular, neutral, post], ignore_index=True)
all_games = all_games.sort_values(["GAME_DATE", "GAME_ID"]).reset_index(drop=True)

pd.crosstab(all_games["SEASON"], all_games["stage"])

stage,Neutral,PlayIn,Playoffs,Regular
SEASON,,,,
2023-24,0,6,82,1230
2024-25,5,6,84,1225
2025-26,5,0,0,1225


## 4. Run Elo game by game

In [6]:
ratings = dict(elo_2223_final)
results = []
season_start_means = {}

for season, season_games in all_games.groupby("SEASON", sort=True):
    # Pull every team part of the way back to the league average
    ratings = {team: CARRYOVER * elo + (1 - CARRYOVER) * LEAGUE_MEAN for team, elo in ratings.items()}
    season_start_means[season] = np.mean(list(ratings.values()))

    for game in season_games.itertuples():
        pre1, pre2 = ratings[game.team1], ratings[game.team2]
        change = elo_change(pre1, pre2, game.margin, game.neutral)
        ratings[game.team1] = pre1 + change
        ratings[game.team2] = pre2 - change
        results.append((game.GAME_ID, pre1, pre2, pre1 + change, pre2 - change,
                        win_prob(pre1, pre2, game.neutral)))

elo = pd.DataFrame(results, columns=["GAME_ID", "ELO1_PRE", "ELO2_PRE", "ELO1_POST", "ELO2_POST", "PROB1"])
games_with_elo = all_games.merge(elo, on="GAME_ID", validate="one_to_one")

## 5. Check the results

A useful check is one that could actually fail if the ratings were wrong.

In [7]:
# Every team starts each season with a league average of 1505
for season, mean in season_start_means.items():
    print(f"{season} league average at the start: {mean:.2f}")
    assert abs(mean - LEAGUE_MEAN) < 0.05

# Elo is zero-sum: what one team gains, the other loses
assert abs(np.mean(list(ratings.values())) - LEAGUE_MEAN) < 0.05

# Game counts: 3,680 regular + 10 neutral + 12 play-in + 166 playoff games
assert len(games_with_elo) == 3680 + 10 + 12 + 166
assert games_with_elo["GAME_ID"].is_unique
assert games_with_elo[["ELO1_PRE", "ELO2_PRE"]].notna().all().all()

# Margins in single_game.csv agree with rest_travel.csv
home_rows = rest_travel[rest_travel["LOCATION_TYPE"] == "Home"].set_index("GAME_ID")["PLUS_MINUS"]
assert (single_game.set_index("GAME_ID")["PLUS_MINUS"] == home_rows).all()

# Each team's next game starts from its last post-game rating, including across seasons
team_rows = pd.concat([
    games_with_elo[["GAME_DATE", "GAME_ID", "SEASON", "team1", "ELO1_PRE", "ELO1_POST"]].set_axis(["date", "id", "season", "team", "pre", "post"], axis=1),
    games_with_elo[["GAME_DATE", "GAME_ID", "SEASON", "team2", "ELO2_PRE", "ELO2_POST"]].set_axis(["date", "id", "season", "team", "pre", "post"], axis=1),
]).sort_values(["team", "date", "id"])
previous_post = team_rows.groupby("team")["post"].shift(1)
new_season = team_rows["season"] != team_rows.groupby("team")["season"].shift(1)
expected = np.where(new_season, CARRYOVER * previous_post + (1 - CARRYOVER) * LEAGUE_MEAN, previous_post)
gaps = (team_rows["pre"] - expected).dropna().abs()
assert gaps.max() < 1e-6

print("PASS: all checks")

2023-24 league average at the start: 1504.97
2024-25 league average at the start: 1504.97
2025-26 league average at the start: 1504.98
PASS: all checks


## 6. Save, keeping the same rows and columns as the first version

In [8]:
regular_elo = games_with_elo.loc[games_with_elo["stage"] == "Regular", ["GAME_ID", "ELO1_PRE", "ELO2_PRE"]]
regular_elo = regular_elo.rename(columns={"ELO1_PRE": "HOME_ELO", "ELO2_PRE": "AWAY_ELO"})

output = single_game.merge(regular_elo, on="GAME_ID", how="left", validate="one_to_one")

assert len(output) == len(single_game)
assert output[["HOME_ELO", "AWAY_ELO"]].notna().all().all()

output.to_csv("single_game_with_elo_corrected.csv", index=False)
print(output.shape)
output[["GAME_DATE", "TEAM_ABBREVIATION_home", "TEAM_ABBREVIATION_away", "PLUS_MINUS", "HOME_ELO", "AWAY_ELO"]].head()

(3680, 14)


,GAME_DATE,TEAM_ABBREVIATION_home,TEAM_ABBREVIATION_away,PLUS_MINUS,HOME_ELO,AWAY_ELO
0,2023-10-27,ATL,NYK,-6,1516.641506,1565.123374
1,2023-10-30,ATL,MIN,14,1530.396733,1525.078488
2,2023-11-01,ATL,WAS,9,1538.769401,1437.070702
3,2023-11-11,ATL,MIA,-8,1559.413675,1561.294090
4,2023-11-15,ATL,NYK,-2,1551.500863,1575.932188


## 7. How much did the ratings change?

In [9]:
old = pd.read_csv("single_game_with_elo.csv")[["GAME_ID", "HOME_ELO", "AWAY_ELO"]]
compare = output.merge(old, on="GAME_ID", suffixes=("_new", "_old"))

for version in ["old", "new"]:
    values = pd.concat([compare[f"HOME_ELO_{version}"], compare[f"AWAY_ELO_{version}"]])
    print(f"{version}: average {values.mean():.1f} | std {values.std():.1f} | range {values.min():.0f} to {values.max():.0f}")

abs_change = pd.concat([(compare["HOME_ELO_new"] - compare["HOME_ELO_old"]).abs(),
                        (compare["AWAY_ELO_new"] - compare["AWAY_ELO_old"]).abs()])
print(f"Average rating change: {abs_change.mean():.1f} | largest: {abs_change.max():.1f}")

old_gap = compare["HOME_ELO_old"] - compare["AWAY_ELO_old"]
new_gap = compare["HOME_ELO_new"] - compare["AWAY_ELO_new"]
print(f"Correlation of old and new rating gaps: {old_gap.corr(new_gap):.3f}")

old: average 1521.2 | std 143.1 | range 1119 to 1895
new: average 1504.9 | std 117.3 | range 1197 to 1814
Average rating change: 33.4 | largest: 123.7
Correlation of old and new rating gaps: 0.978


In [10]:
# Prediction check on regular-season games with a home team (lower Brier score is better)
home_won = (compare["PLUS_MINUS"] > 0).astype(int)
prob_new = win_prob(compare["HOME_ELO_new"], compare["AWAY_ELO_new"])
prob_old = win_prob(compare["HOME_ELO_old"], compare["AWAY_ELO_old"])

print("Brier score, old:", round(((prob_old - home_won) ** 2).mean(), 4))
print("Brier score, new:", round(((prob_new - home_won) ** 2).mean(), 4))
print("Predicted home win rate:", round(prob_new.mean(), 3), "| actual:", round(home_won.mean(), 3))

Brier score, old: 0.216
Brier score, new: 0.2146
Predicted home win rate: 0.619 | actual: 0.547


In [11]:
# Ratings after the 2025-26 regular season
final_table = pd.DataFrame({"Corrected Elo": pd.Series(ratings)}).sort_values("Corrected Elo", ascending=False).round(0)
final_table.index.name = "Team"
final_table.head(10)

,Corrected Elo
Team,
SAS,1749.0
OKC,1707.0
DET,1703.0
BOS,1701.0
DEN,1665.0
CHA,1656.0
NYK,1656.0
CLE,1619.0
HOU,1615.0
